In [6]:
import numpy as np
import pandas as pd
import torch
import os
from datasets import load_from_disk, load_dataset


In [10]:
# Cell 1 - Load and inspect Excel thoroughly
import pandas as pd
import os

df = pd.read_excel("../data/vqa_rad_full/VQA_RAD Dataset Public.xlsx", )
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nFirst 3 rows:")
df.head(3)

Shape: (2248, 14)

Columns: ['QID_unique', 'QID_para', 'QID_linked', 'IMAGEID_case', 'IMAGEID', 'IMAGEORGAN', 'EVALUATION', 'QUESTION', 'Q_REPHASE', 'Q_RELATION', 'Q_FRAMED', 'Q_TYPE', 'ANSWER', 'A_TYPE']

Dtypes:
QID_unique       int64
QID_para           str
QID_linked         str
IMAGEID_case       str
IMAGEID            str
IMAGEORGAN         str
EVALUATION         str
QUESTION           str
Q_REPHASE          str
Q_RELATION         str
Q_FRAMED           str
Q_TYPE             str
ANSWER          object
A_TYPE             str
dtype: object

First 3 rows:


/ix/cs2770_2026s/abn80/conda/envs/medvlm/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,QID_unique,QID_para,QID_linked,IMAGEID_case,IMAGEID,IMAGEORGAN,EVALUATION,QUESTION,Q_REPHASE,Q_RELATION,Q_FRAMED,Q_TYPE,ANSWER,A_TYPE
0,0,freeform,03f451ca-de62-4617-9679-e836026a7642,https://medpix.nlm.nih.gov/case?id=48e1dd0e-85...,https://medpix.nlm.nih.gov/images/full/synpic5...,HEAD,not evaluated,Are regions of the brain infarcted?,NaN,NaN,NaN,PRES,Yes,CLOSED
1,1,freeform,06e26b2c-04b9-42bc-8e98-1de30a0f7682,https://medpix.nlm.nih.gov/case?id=b197277b-69...,https://medpix.nlm.nih.gov/images/full/synpic2...,CHEST,not evaluated,Are the lungs normal appearing?,NaN,NaN,NaN,ABN,No,CLOSED
2,2,freeform,0d0e8b6b-7753-4788-9b6d-dc7f25250c3f,https://medpix.nlm.nih.gov/case?id=b197277b-69...,https://medpix.nlm.nih.gov/images/full/synpic2...,CHEST,not evaluated,Is there evidence of a pneumothorax,NaN,NaN,NaN,PRES,No,CLOSED


In [11]:
# Cell 2 - Check every column
for col in df.columns:
    print(f"\n{'='*60}")
    print(f"=== {col} ===")
    print(f"  Nulls/NaN: {df[col].isna().sum()}")
    print(f"  Unique: {df[col].nunique()}")
    if df[col].nunique() < 25:
        print(f"  Values:\n{df[col].value_counts().to_string()}")
    else:
        print(f"  Top 10:\n{df[col].value_counts().head(10).to_string()}")
        print(f"  Sample: {df[col].head(5).tolist()}")


=== QID_unique ===
  Nulls/NaN: 0
  Unique: 2248
  Top 10:
QID_unique
0    1
1    1
2    1
3    1
4    1
5    1
6    1
7    1
8    1
9    1
  Sample: [0, 1, 2, 3, 4]

=== QID_para ===
  Nulls/NaN: 0
  Unique: 4
  Values:
QID_para
freeform         1206
para              591
test_freeform     308
test_para         143

=== QID_linked ===
  Nulls/NaN: 0
  Unique: 1453
  Top 10:
QID_linked
c5e00888-d92b-4ee0-892c-92edc117687c    6
b8e5d153-03dc-4a54-aad1-408af7c123ce    6
44e26c7f-b5ce-4cc3-b1be-37dc87f7e15c    6
76ef2388-3984-4788-84c4-1c4c4202b109    6
4bb981e1-73b0-437b-96f9-b186ebe9f162    5
5d08704a-3ebf-452c-a489-eb8f8c06448b    4
1db8f430-ad44-48fa-a802-046c92134729    3
b5bc8692-7599-4454-92c6-e0268c6d390f    3
00ae02fb-b953-4e4f-a8cb-3e3b2672fd3a    2
013b2c30-b7f1-4ce4-be48-ecf63316b983    2
  Sample: ['03f451ca-de62-4617-9679-e836026a7642', '06e26b2c-04b9-42bc-8e98-1de30a0f7682', '0d0e8b6b-7753-4788-9b6d-dc7f25250c3f', '0e90b6bc-265f-490b-a039-509b9907a3cb', '1179f612-12e0-4dda

In [12]:
# Cell 3 - Check images
import os
img_dir = "../data/vqa_rad_full/images"
disk_images = set(os.listdir(img_dir))
ref_images = set(df['IMAGEID'].dropna().unique())

print(f"Images on disk: {len(disk_images)}")
print(f"Images referenced: {len(ref_images)}")
print(f"On disk but not in data: {disk_images - ref_images}")
print(f"In data but not on disk: {ref_images - disk_images}")

Images on disk: 315
Images referenced: 314
On disk but not in data: {'synpic47974.jpg', 'synpic58261.jpg', 'synpic29795.jpg', 'synpic22156.jpg', 'synpic40464.jpg', 'synpic33331.jpg', 'synpic53207.jpg', 'synpic55583.jpg', 'synpic34922.jpg', 'synpic100228.jpg', 'synpic50962.jpg', 'synpic57317.jpg', 'synpic28180.jpg', 'synpic54802.jpg', 'synpic53033.jpg', 'synpic56116.jpg', 'synpic22097.jpg', 'synpic27597.jpg', 'synpic24967.jpg', 'synpic21995.jpg', 'synpic52282.jpg', 'synpic25534.jpg', 'synpic22828.jpg', 'synpic15006.jpg', 'synpic60254.jpg', 'synpic34449.jpg', 'synpic46943.jpg', 'synpic676.jpg', 'synpic20208.jpg', 'synpic51383.jpg', 'synpic45699.jpg', 'synpic27402.jpg', 'synpic33302.jpg', 'synpic29263.jpg', 'synpic57935.jpg', 'synpic22286.jpg', 'synpic43648.jpg', 'synpic55245.jpg', 'synpic29265.jpg', 'synpic49914.jpg', 'synpic52951.jpg', 'synpic34836.jpg', 'synpic20260.jpg', 'synpic34017.jpg', 'synpic51872.jpg', 'synpic27277.jpg', 'synpic53228.jpg', 'synpic28355.jpg', 'synpic34854.jpg', '

In [13]:
# Cell 4 - Check duplicates
dupes = df.duplicated(subset=['IMAGEID', 'QUESTION', 'ANSWER'], keep=False)
print(f"Duplicate rows: {dupes.sum()}")
if dupes.sum() > 0:
    print(df[dupes][['IMAGEID', 'QUESTION', 'ANSWER', 'QID_para']].to_string())

Duplicate rows: 8
                                                    IMAGEID                                             QUESTION                ANSWER   QID_para
78   https://medpix.nlm.nih.gov/images/full/synpic16174.jpg  Is the descending aortic silhouette of normal size?                   Yes   freeform
79   https://medpix.nlm.nih.gov/images/full/synpic16174.jpg  Is the descending aortic silhouette of normal size?                   Yes       para
195  https://medpix.nlm.nih.gov/images/full/synpic32933.jpg                    Is the left hemidiaphragm normal?                   Yes       para
196  https://medpix.nlm.nih.gov/images/full/synpic32933.jpg                    Is the left hemidiaphragm normal?                   Yes   freeform
422  https://medpix.nlm.nih.gov/images/full/synpic31955.jpg                         Where is the lesion located?  Anterior mediastinum       para
447  https://medpix.nlm.nih.gov/images/full/synpic31955.jpg                         Where is the lesion lo

In [14]:
# Cell 5 - Check ANSWER column types (it's object, might have mixed types)
print(f"ANSWER dtype: {df['ANSWER'].dtype}")
print(f"\nNon-string answers:")
non_str = df[df['ANSWER'].apply(lambda x: not isinstance(x, str))]
print(f"  Count: {len(non_str)}")
if len(non_str) > 0:
    print(non_str[['QUESTION', 'ANSWER', 'Q_TYPE', 'A_TYPE']].head(10))

ANSWER dtype: object

Non-string answers:
  Count: 7
                                               QUESTION ANSWER Q_TYPE A_TYPE
334              What lesions are present in the lungs?    NaN   PRES   OPEN
1511                How many gallstones are identified?      4  COUNT   OPEN
1568   How many kidneys are visualizable in this image?      2  COUNT   OPEN
1683                How many kidneys are in this image?      2  COUNT   OPEN
2161  About how often do you see bilateral Wilms tumor?   0.05  OTHER   OPEN
2234  How many ribs are superimposed on the lung fie...     12  COUNT   OPEN
2235  How many ribs are present in vertical order on...     12  COUNT   OPEN


In [15]:
# Cell - FULL CLEANING PIPELINE
import pandas as pd
import os

df = pd.read_excel("../data/vqa_rad_full/VQA_RAD Dataset Public.xlsx")
print(f"Original shape: {df.shape}")

# 1. Rename columns to clean lowercase names
df = df.rename(columns={
    'QID_unique': 'qid',
    'QID_para': 'phrase_type',
    'QID_linked': 'qid_linked',
    'IMAGEID_case': 'case_url',
    'IMAGEID': 'image_url',
    'IMAGEORGAN': 'image_organ',
    'EVALUATION': 'evaluation',
    'QUESTION': 'question',
    'Q_REPHASE': 'question_rephrased',
    'Q_RELATION': 'question_relation',
    'Q_FRAMED': 'question_framed',
    'Q_TYPE': 'question_type',
    'ANSWER': 'answer',
    'A_TYPE': 'answer_type'
})

# 2. Extract image filename from URL
df['image_name'] = df['image_url'].str.split('/').str[-1]
print(f"Unique images: {df['image_name'].nunique()}")

# 3. Fix answer_type inconsistency ("CLOSED  " with extra space -> "CLOSED")
df['answer_type'] = df['answer_type'].str.strip().str.upper()
print(f"Answer types after fix: {df['answer_type'].value_counts().to_dict()}")

# 4. Convert all answers to string, handle NaN
print(f"\nNull answers: {df['answer'].isna().sum()}")
df['answer'] = df['answer'].fillna('unanswerable').astype(str)

# 5. Normalize answer casing (lowercase for consistency)
df['answer_normalized'] = df['answer'].str.strip().str.lower()

# 6. Clean question_type - fix typos and multi-labels
df['question_type_raw'] = df['question_type']  # keep original
df['question_type'] = df['question_type'].str.strip()

# Fix known typos
df['question_type'] = df['question_type'].replace({
    'ATRIB': 'ATTRIB',
    'Other': 'OTHER'
})

# Create primary question type (first label for multi-labels)
df['question_type_primary'] = df['question_type'].str.split(',').str[0].str.strip()
df['question_type_primary'] = df['question_type_primary'].replace({
    'ATRIB': 'ATTRIB',
    'Other': 'OTHER'
})

print(f"\nQuestion types (primary): {df['question_type_primary'].value_counts().to_dict()}")

# 7. Create split column
df['split'] = df['phrase_type'].map({
    'freeform': 'train',
    'para': 'train',
    'test_freeform': 'test',
    'test_para': 'test'
})
print(f"\nSplit: {df['split'].value_counts().to_dict()}")

# 8. Remove duplicates (keep first occurrence)
before = len(df)
df = df.drop_duplicates(subset=['image_name', 'question', 'answer'], keep='first')
print(f"\nDropped {before - len(df)} duplicates, remaining: {len(df)}")

# 9. Verify images exist on disk
img_dir = "../data/vqa_rad_full/images"
disk_images = set(os.listdir(img_dir))
df['image_exists'] = df['image_name'].isin(disk_images)
print(f"Images found on disk: {df['image_exists'].sum()}/{len(df)}")

# 10. Final summary
print(f"\n{'='*60}")
print(f"FINAL DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total QA pairs: {len(df)}")
print(f"Unique images: {df['image_name'].nunique()}")
print(f"Train: {(df['split']=='train').sum()}")
print(f"Test: {(df['split']=='test').sum()}")
print(f"Closed: {(df['answer_type']=='CLOSED').sum()}")
print(f"Open: {(df['answer_type']=='OPEN').sum()}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

Original shape: (2248, 14)
Unique images: 314
Answer types after fix: {'CLOSED': 1299, 'OPEN': 949}

Null answers: 1

Question types (primary): {'PRES': 812, 'POS': 324, 'ABN': 205, 'OTHER': 196, 'MODALITY': 185, 'SIZE': 175, 'PLANE': 120, 'ATTRIB': 93, 'ORGAN': 59, 'COLOR': 54, 'COUNT': 24, 'PRSE': 1}

Split: {'train': 1797, 'test': 451}

Dropped 4 duplicates, remaining: 2244
Images found on disk: 2244/2244

FINAL DATASET SUMMARY
Total QA pairs: 2244
Unique images: 314
Train: 1794
Test: 450
Closed: 1297
Open: 947

Columns: ['qid', 'phrase_type', 'qid_linked', 'case_url', 'image_url', 'image_organ', 'evaluation', 'question', 'question_rephrased', 'question_relation', 'question_framed', 'question_type', 'answer', 'answer_type', 'image_name', 'answer_normalized', 'question_type_raw', 'question_type_primary', 'split', 'image_exists']


/ix/cs2770_2026s/abn80/conda/envs/medvlm/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


,qid,phrase_type,qid_linked,case_url,image_url,image_organ,evaluation,question,question_rephrased,question_relation,question_framed,question_type,answer,answer_type,image_name,answer_normalized,question_type_raw,question_type_primary,split,image_exists
0,0,freeform,03f451ca-de62-4617-9679-e836026a7642,https://medpix.nlm.nih.gov/case?id=48e1dd0e-85...,https://medpix.nlm.nih.gov/images/full/synpic5...,HEAD,not evaluated,Are regions of the brain infarcted?,NaN,NaN,NaN,PRES,Yes,CLOSED,synpic54610.jpg,yes,PRES,PRES,train,True
1,1,freeform,06e26b2c-04b9-42bc-8e98-1de30a0f7682,https://medpix.nlm.nih.gov/case?id=b197277b-69...,https://medpix.nlm.nih.gov/images/full/synpic2...,CHEST,not evaluated,Are the lungs normal appearing?,NaN,NaN,NaN,ABN,No,CLOSED,synpic29265.jpg,no,ABN,ABN,train,True
2,2,freeform,0d0e8b6b-7753-4788-9b6d-dc7f25250c3f,https://medpix.nlm.nih.gov/case?id=b197277b-69...,https://medpix.nlm.nih.gov/images/full/synpic2...,CHEST,not evaluated,Is there evidence of a pneumothorax,NaN,NaN,NaN,PRES,No,CLOSED,synpic29265.jpg,no,PRES,PRES,train,True
3,3,freeform,0e90b6bc-265f-490b-a039-509b9907a3cb,https://medpix.nlm.nih.gov/case?id=19aa8a2b-35...,https://medpix.nlm.nih.gov/images/full/synpic2...,CHEST,given,What type of imaging does this not represent?,NaN,NaN,NaN,MODALITY,ultrasound,OPEN,synpic28602.jpg,ultrasound,MODALITY,MODALITY,train,True
4,4,freeform,1179f612-12e0-4dda-aee0-f14a5200be7b,https://medpix.nlm.nih.gov/case?id=b197277b-69...,https://medpix.nlm.nih.gov/images/full/synpic2...,CHEST,given,Is this a MRI of the chest?,NaN,NaN,NaN,MODALITY,no,CLOSED,synpic29265.jpg,no,MODALITY,MODALITY,train,True


In [16]:
# Cell - Fix PRSE typo
df['question_type_primary'] = df['question_type_primary'].replace({'PRSE': 'PRES'})
print(f"Question types (primary): {df['question_type_primary'].value_counts().to_dict()}")

Question types (primary): {'PRES': 813, 'POS': 322, 'ABN': 204, 'OTHER': 196, 'MODALITY': 185, 'SIZE': 174, 'PLANE': 120, 'ATTRIB': 93, 'ORGAN': 59, 'COLOR': 54, 'COUNT': 24}


In [17]:
# Cell - Prepare for HuggingFace upload
from datasets import Dataset, DatasetDict, Image as HFImage, Features, Value
from PIL import Image
import os

img_dir = "../data/vqa_rad_full/images"

# Select columns for the HF dataset
hf_columns = [
    'qid', 'image_name', 'image_organ', 'question', 'answer', 'answer_normalized',
    'answer_type', 'question_type_primary', 'question_type_raw', 'phrase_type',
    'evaluation', 'split'
]

df_hf = df[hf_columns].copy()

# Add image paths
df_hf['image'] = df_hf['image_name'].apply(lambda x: os.path.join(img_dir, x))

# Verify all images load
bad = []
for idx, row in df_hf.iterrows():
    try:
        img = Image.open(row['image'])
        img.verify()
    except Exception as e:
        bad.append((row['image_name'], str(e)))
print(f"Bad images: {len(bad)}")
if bad:
    print(bad)

# Split into train/test
train_df = df_hf[df_hf['split'] == 'train'].reset_index(drop=True)
test_df = df_hf[df_hf['split'] == 'test'].reset_index(drop=True)

print(f"Train: {len(train_df)}, Test: {len(test_df)}")

# Create HF datasets
train_ds = Dataset.from_pandas(train_df).cast_column("image", HFImage())
test_ds = Dataset.from_pandas(test_df).cast_column("image", HFImage())

ds = DatasetDict({
    'train': train_ds,
    'test': test_ds
})

print(ds)
print(f"\nFeatures: {ds['train'].features}")

Bad images: 0
Train: 1794, Test: 450
DatasetDict({
    train: Dataset({
        features: ['qid', 'image_name', 'image_organ', 'question', 'answer', 'answer_normalized', 'answer_type', 'question_type_primary', 'question_type_raw', 'phrase_type', 'evaluation', 'split', 'image'],
        num_rows: 1794
    })
    test: Dataset({
        features: ['qid', 'image_name', 'image_organ', 'question', 'answer', 'answer_normalized', 'answer_type', 'question_type_primary', 'question_type_raw', 'phrase_type', 'evaluation', 'split', 'image'],
        num_rows: 450
    })
})

Features: {'qid': Value('int64'), 'image_name': Value('string'), 'image_organ': Value('large_string'), 'question': Value('large_string'), 'answer': Value('large_string'), 'answer_normalized': Value('large_string'), 'answer_type': Value('large_string'), 'question_type_primary': Value('string'), 'question_type_raw': Value('large_string'), 'phrase_type': Value('large_string'), 'evaluation': Value('large_string'), 'split': Value('l

In [18]:
# Cell - Verify a sample looks correct
sample = ds['train'][0]
print(f"Question: {sample['question']}")
print(f"Answer: {sample['answer']}")
print(f"Answer (normalized): {sample['answer_normalized']}")
print(f"Answer type: {sample['answer_type']}")
print(f"Question type: {sample['question_type_primary']}")
print(f"Image organ: {sample['image_organ']}")
print(f"Image: {sample['image'].size}")

Question: Are regions of the brain infarcted?
Answer: Yes
Answer (normalized): yes
Answer type: CLOSED
Question type: PRES
Image organ: HEAD
Image: (566, 555)


In [19]:
# Cell - Login and push
from huggingface_hub import login
import os

login(token=os.environ['HF_TOKEN'])

# Push to HF - what's your HF username?
ds.push_to_hub(
    "abhay2812/vqa-rad-full",
    private=False,
    commit_message="VQA-RAD full dataset with question types, answer types, image organ, and evaluation metadata"
)
print("Pushed successfully!")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Map:   0%|          | 0/1794 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Pushed successfully!


In [1]:
import torch

print(f"Model device: {next(model.parameters()).device}")
print(f"PyTorch allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")

NameError: name 'model' is not defined